# Notebook 8b — Does Eq. 15 collapse the representations? (no training, about 10 min on CPU)

**Why this notebook.** In Notebook 8, A0 (M2 + Eq. 15, published form) lost 15–18 pp of macro-F1 on MOSEI whenever text was
missing, with no change when all inputs were present. Its training alignment term fell to about 0.05 on MOSEI (MOSI: 0.36–0.86).
**Hypothesis:** Eq. 15 pushes audio/video representations so close to the text direction that they collapse onto a shared
direction, and this coincides with near-constant predictions without text. This notebook **measures** that. It trains no model.
The only fitted quantity is a closed-form least-squares read-out (one matrix solve, no iterations, fixed penalty, < 1 s).

**Models compared** (3 seeds each, saved checkpoints): **A0 vs M2** (main pair) and **B6 vs M4** (Eq. 15 on available pairs, secondary).

**Measurements and what they mean** (z_m = the representation Eq. 15 acts on: the pooled projection of modality m, inputs all present)
| Measurement | Definition | Collapse looks like |
|---|---|---|
| Text–modality cosine | mean over clips of cos(z_t, z_m) | close to 1 |
| Bias norm | ‖projection bias‖ (M2/A0: Linear bias; M4/B6: LayerNorm bias) | large vs spread |
| Common-to-specific ratio R | ‖mean of z_m‖ / spread, spread = √(total variance across clips) | large: every clip points the same way |
| Pairwise cosine | mean cos(z_m^i, z_m^j) between different clips | close to 1 |
| Effective rank | exp(entropy of the normalised covariance eigenvalues), 1…128 | small: variation squeezed into few directions |
| Read-out r | Pearson r between the true sentiment score and a least-squares read-out from standardised z_m (fit on train, scored on validation) | low: sentiment information lost |
| Predicted-class shares | share of each predicted class without text (test, from the saved Notebook 8 predictions) | one class ≥ 90 % |
| Classifier input (h) | the same R / read-out on the vector the classifier receives, under A, V, A+V and T+A+V; plus the gate weights | — |
| Own-output ranking r | Pearson r between the model's own (positive − negative) logit and the true score, per input combination (no fitting) | low: the model's outputs no longer track sentiment |

**Criteria fixed before looking (MOSEI, A0 vs M2, every seed must satisfy):**
- **K1 strong alignment:** A0's cos(z_t, z_a) and cos(z_t, z_v) ≥ 0.90.
- **K2 directional collapse:** A0's ratio R ≥ 2 × M2's, for audio and video.
- **K3 dimensional collapse:** A0's effective rank ≤ 0.5 × M2's, for audio and video.
- **K4 information loss:** A0's read-out r ≤ 0.8 × M2's, for audio and video (only evaluated where M2's r ≥ 0.10).
- **K5 prediction collapse:** A0's majority predicted class ≥ 90 % of test clips under A, V and A+V (mean of seeds), and M2's is below 90 %.
- **K6 ranking kept:** A0's own-output ranking r ≥ 0.8 × M2's under each of A, V, A+V (validation, mean of seeds; only where M2's r ≥ 0.10).
- Caution on K4 (noticed in the sandbox test, before any MOSEI result was seen): in M2/A0, z_m is a *linear* function of the pooled
  input, so a linear read-out can only lose information if the projection becomes nearly singular. K4 therefore tests only that.
  K6 does not have this blind spot.
- Wording rules: if K4 fails, the notebook says "substantial information loss was not established" (not "information is preserved");
  "not evaluable" cases are reported separately. K6 is decided on Pearson r; Spearman ρ is reported next to it whenever ranking is discussed.

**Reading (printed automatically):**
- K1 fails → alignment is not strong; the hypothesis does not apply.
- K1 and (K2 or K3) and K4 and K5 → consistent with a collapse that removes audio/video sentiment information.
- K1 and (K2 or K3) and K5 but **not** K4 → representations collapse in direction/dimension, but the information is still linearly
  recoverable, so information loss in z does **not** explain the near-constant predictions (look at the classifier input).
- K1 but neither K2 nor K3 → no collapse found; the hypothesis is not supported.
- Anything else → mixed; reported as such.
- Added clause: if K5 and K6 both hold → the outputs still rank clips by sentiment, so the near-constant predictions are consistent with
  an **output shift towards one class** (an offset problem), not with lost ranking information.

**What this cannot show:** cause. Every result here is an **association** between Eq. 15 and the measured properties in 3 seeds.
A causal test would vary λ1 and retrain — not done (no new training, by decision). MOSI is shown for comparison (weaker alignment).

**Reads** `v8_ablations/` and the reference folders; **writes only** `v8_ablations/eq15_check/`. Runtime: **CPU**, High-RAM.

In [1]:
import os, sys, time, json, random, pickle, platform
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import spearmanr
DEVICE = "cpu"

Mounted at /content/drive


## 1. Settings

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
PATHS = {"MOSI": "/content/drive/MyDrive/datasets/CMU-MOSI/Processed/aligned_50.pkl",
         "MOSEI": "/content/drive/MyDrive/datasets/CMU-MOSEI/aligned_50.pkl"}
REF_DIR = {"MOSEI": f"{BASE_OUT}/mosei_v2_cpu", "MOSI": f"{BASE_OUT}/mosi_v7_confirmation"}
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]
    PATHS = {"MOSI": os.environ["EMOFAR_MOSI"], "MOSEI": os.environ["EMOFAR_MOSEI"]}
    REF_DIR = {"MOSEI": os.environ["EMOFAR_REF_MOSEI"], "MOSI": os.environ["EMOFAR_REF_MOSI"]}
V8 = f"{BASE_OUT}/v8_ablations"
OUT_DIR = f"{V8}/eq15_check"; os.makedirs(OUT_DIR, exist_ok=True)
v2cfg = json.load(open(f"{REF_DIR['MOSEI']}/checkpoints/M4_seed42_log.json"))["train_config"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (v2cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
SEEDS = [42, 43, 44]
THREADS = 4 if not TEST_MODE else int(os.environ.get("EMOFAR_THREADS", 2)); torch.set_num_threads(THREADS)
LAMBDA1 = 0.3
RIDGE = 1.0                  # fixed penalty for the least-squares read-out (not tuned)
H_TRAIN_MAX = 6000           # clips used to fit the read-out on the classifier input (fixed random subset of train)
PAIRS = [("A0", "M2"), ("B6", "M4")]
REP_MODELS = ["M2", "A0", "M4", "B6"]
ALL = ["M2", "M3", "M4", "A0", "A1", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "B5", "B6"]
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]
CONDS = ["T", "A", "V", "A+V", "T+A", "T+V", "T+A+V"]
NO_TEXT = ["A", "V", "A+V"]
ROW = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}
print({"torch": torch.__version__, "threads": torch.get_num_threads(), "ridge": RIDGE, "seeds": SEEDS})

{'torch': '2.14.1+cpu', 'threads': 4, 'ridge': 1.0, 'seeds': [42, 43, 44]}


## 2. Data loading, model code (identical to Notebook 8)

In [3]:
DIMS = None
def load_data(DATA_PATH, RUN_DIR):
    global DIMS
    t0 = time.time()
    with open(DATA_PATH, "rb") as f:
        raw = pickle.load(f)
    print(f"Loaded in {time.time() - t0:.0f}s")

    def prep(split):
        d = raw[split]
        mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
        feats = {"text": d["text"].astype(np.float32),
                 "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
                 "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
        score = d["regression_labels"].astype(np.float32)
        y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
        return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

    data = {s: prep(s) for s in ["train", "valid", "test"]}
    del raw

    for m in ["audio", "vision"]:                                           # standardise with train statistics
        tr = data["train"]["feats"][m][data["train"]["mask"]]
        mu, sd = tr.mean(0), tr.std(0) + 1e-6
        for s in data:
            x = (data[s]["feats"][m] - mu) / sd
            data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

    DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
    print("Feature sizes:", DIMS)
    for s in data:
        counts = np.bincount(data[s]["y"], minlength=3)
        print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
    np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
    np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
    pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
    return data

def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

class RMSRVar(PaperRMSR):
    """M2 with optional gate masking (A1, A3) or M4's 2-layer classifier (A4). Can return z for Eq. 15 (A0)."""
    def __init__(self, gate_mask=False, two_layer_cls=False):
        super().__init__()
        self.gate_mask = gate_mask
        if two_layer_cls:
            self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False, return_z=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}                         # Eq. 10
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)       # Eq. 11
        if self.gate_mask:
            q = q.masked_fill(avail == 0, float("-inf"))                                      # missing input -> weight 0
        gamma = q.softmax(-1)                                                                 # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                         # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        if return_z:
            return out, z
        return (out, gamma) if return_gate else out

class FFVar(FixedFusion):
    """M4 with one component switched: gate masking (B1), Eq. 11-12 gate (B2), pooled attention (B3),
    1-layer classifier (B4), no LayerNorm (B5). Can return z for Eq. 15 (B6)."""
    def __init__(self, gate_mask=True, gate="linear", attention="token", classifier="two_layer", layernorm=True):
        super().__init__()
        self.gate_mask, self.gate_type, self.attention = gate_mask, gate, attention
        if not layernorm:                       # LayerNorm has no random initial values: the Linear weights stay identical
            for m in MODS:
                self.proj[m][1] = nn.Identity()
        if gate == "rmsr":                      # Eq. 11-12 on the per-modality representations h_m
            del self.gate
            self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})
        if attention == "pooled":               # M2-style: W_v applied to the pooled projection, plus M4's residual
            del self.xattn
            self.wv = nn.ModuleDict({m: nn.Linear(D_MODEL, D_MODEL) for m in ["audio", "vision"]})
        if classifier == "one_layer":
            self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False, return_z=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            if self.attention == "token":
                attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
                h[m] = masked_mean(attended + p[m], mask)
            else:
                pooled = masked_mean(p[m], mask)
                h[m] = self.wv[m](pooled) + pooled
        if self.gate_type == "linear":
            logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        else:
            logits = torch.stack([torch.sigmoid((h[m] * self.u[m]).sum(-1)) for m in MODS], -1)
        if self.gate_mask:
            logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        if return_z:
            return out, {m: masked_mean(p[m], mask) for m in MODS}
        return (out, w) if return_gate else out

SPECS = {
    "M2": dict(family="ref_rmsr", arch={}, drop=0.0, eq15=None),
    "M3": dict(family="ref_ff", arch={}, drop=0.0, eq15=None),
    "M4": dict(family="ref_ff", arch={}, drop=MOD_DROPOUT, eq15=None),
    "A0": dict(family="rmsr", arch={}, drop=0.0, eq15="published"),
    "A1": dict(family="rmsr", arch={"gate_mask": True}, drop=0.0, eq15=None, weights_from="M2"),
    "A2": dict(family="rmsr", arch={}, drop=MOD_DROPOUT, eq15=None),
    "A3": dict(family="rmsr", arch={"gate_mask": True}, drop=MOD_DROPOUT, eq15=None),
    "A4": dict(family="rmsr", arch={"two_layer_cls": True}, drop=0.0, eq15=None),
    "B1": dict(family="ff", arch={"gate_mask": False}, drop=MOD_DROPOUT, eq15=None),
    "B2": dict(family="ff", arch={"gate": "rmsr"}, drop=MOD_DROPOUT, eq15=None),
    "B3": dict(family="ff", arch={"attention": "pooled"}, drop=MOD_DROPOUT, eq15=None),
    "B4": dict(family="ff", arch={"classifier": "one_layer"}, drop=MOD_DROPOUT, eq15=None),
    "B5": dict(family="ff", arch={"layernorm": False}, drop=MOD_DROPOUT, eq15=None),
    "B6": dict(family="ff", arch={}, drop=MOD_DROPOUT, eq15="valid_pairs"),
}
REF_OF = {v: ("M2" if SPECS[v]["family"] == "rmsr" else "M4") for v in SPECS if not v.startswith("M")}
REFS = ["M2", "M3", "M4"]
TRAINED = ["A0", "A2", "A3", "A4", "B1", "B2", "B3", "B4", "B5", "B6"]
COMPARISONS = [("C01", "Eq. 15 (published) on M2", "A0", "M2"),
               ("C02", "Gate masking on M2, no dropout (test-time only)", "A1", "M2"),
               ("C03", "Modality dropout on M2", "A2", "M2"),
               ("C04", "Masking + dropout on M2 (TWO changes)", "A3", "M2"),
               ("C05", "Gate masking on M2 when dropout is used", "A3", "A2"),
               ("C06", "2-layer classifier on M2", "A4", "M2"),
               ("C07", "Modality dropout on M3", "M4", "M3"),
               ("C08", "Gate masking in M4", "M4", "B1"),
               ("C09", "Linear gate vs Eq. 11-12 gate", "M4", "B2"),
               ("C10", "Token attention vs pooled attention", "M4", "B3"),
               ("C11", "2-layer vs 1-layer classifier", "M4", "B4"),
               ("C12", "LayerNorm in the projection", "M4", "B5"),
               ("C13", "Eq. 15 (available pairs) on M4", "B6", "M4")]

def build(vid):
    sp = SPECS[vid]
    if sp["family"] == "ref_rmsr":
        return PaperRMSR()
    if sp["family"] == "ref_ff":
        return FixedFusion()
    return (RMSRVar if sp["family"] == "rmsr" else FFVar)(**sp["arch"])

def align_loss(z, avail, valid_only):
    """Eq. 15: sum over audio and video of (1 - cos(z_text, z_m)), divided by the batch size (same scale as the cross-entropy).
    valid_only=False -> published form, every sample and pair. valid_only=True -> a pair counts only if text and that modality
    are both present; a sample with no valid pair adds 0. Returns (loss, number of samples with no valid pair)."""
    B = avail.shape[0]; total = z["text"].new_zeros(()); no_pair = torch.ones(B, dtype=torch.bool)
    for i, m in [(1, "audio"), (2, "vision")]:
        ok = ((avail[:, 0] > 0) & (avail[:, i] > 0)) if valid_only else torch.ones(B, dtype=torch.bool)
        no_pair &= ~ok
        if ok.any():
            total = total + (1 - F.cosine_similarity(z["text"][ok], z[m][ok], dim=-1)).sum()
    return total / B, int(no_pair.sum())

def train_variant(vid, seed):
    """v2's train(), with two additions: the model comes from SPECS, and Eq. 15 is added to the loss when the spec asks for it.
    For a spec without Eq. 15 the operations are exactly those of train() (checked bit-for-bit in Section 6)."""
    sp = SPECS[vid]; drop, eq15 = sp["drop"], sp["eq15"]
    set_seed(seed)
    model = build(vid).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        s_ce, s_al, n_b, n_nopair = 0.0, 0.0, 0, 0
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            yb = torch.from_numpy(tr["y"][idx]).to(DEVICE)
            if eq15 is None:
                loss = F.cross_entropy(model(x, mask, avail), yb, weight=class_w); ce = loss
            else:
                out, z = model(x, mask, avail, return_z=True)
                ce = F.cross_entropy(out, yb, weight=class_w)
                al, nn_ = align_loss(z, avail, valid_only=(eq15 == "valid_pairs"))
                loss = ce + LAMBDA1 * al
                s_al += al.item(); n_nopair += nn_
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            s_ce += ce.item(); n_b += 1
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        hist.append({"epoch": ep + 1, "ce": s_ce / n_b, "align": (s_al / n_b) if eq15 else None,
                     "frac_no_valid_pair": (n_nopair / len(order)) if eq15 else None, "valid_macro_f1": float(val)})
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, float(best), ep + 1, hist

@torch.no_grad()
def predict_cond(model, d, cond):
    """Same as v2's predict(), but with a fixed modality combination for every clip."""
    model.eval(); out = []; row = np.array(ROW[cond], dtype=np.float32)
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, np.tile(row, (len(idx), 1)))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

## 3. Measurement functions

In [4]:
def ckpt_path(ds, vid, s):
    reuse = json.load(open(f"{V8}/{ds.lower()}/reference_check.json"))["reuse"]
    if vid in ("M2", "M3", "M4") and reuse:
        return f"{REF_DIR[ds]}/checkpoints/{vid}_seed{s}.pt"
    return f"{V8}/{ds.lower()}/checkpoints/{vid}_seed{s}.pt"

def load_model(ds, vid, s):
    """M2/A0 -> RMSRVar(), M4/B6 -> FFVar(): the same networks as in Notebook 8 (identity checked in its code check),
    used here because they can return z. Loaded outputs are checked against the saved Notebook 8 predictions."""
    m = RMSRVar() if vid in ("M2", "A0") else FFVar()
    m.load_state_dict(torch.load(ckpt_path(ds, vid, s), map_location="cpu")); m.eval()
    return m

@torch.no_grad()
def z_all(model, d, bs=512):
    """z_m for every clip, all inputs present: exactly what Eq. 15 acts on (checked against return_z below)."""
    out = {m: [] for m in MODS}
    for i in range(0, len(d["y"]), bs):
        idx = np.arange(i, min(i + bs, len(d["y"]))); mask = torch.from_numpy(d["mask"][idx])
        for m in MODS:
            x = torch.from_numpy(d["feats"][m][idx])
            z = masked_mean(model.proj[m](x), mask) if isinstance(model, FFVar) else model.proj[m](masked_mean(x, mask))
            out[m].append(z.double().numpy())
    return {m: np.concatenate(v) for m, v in out.items()}

@torch.no_grad()
def forward_cond(model, d, cond, idx_all=None, bs=512):
    """Classifier input h, logits and gate weights under one input combination."""
    cap = {}; hook = model.cls.register_forward_hook(lambda mod, inp, out: cap.__setitem__("h", inp[0]))
    idx_all = np.arange(len(d["y"])) if idx_all is None else idx_all
    H, L, G = [], [], []; row = np.array(ROW[cond], dtype=np.float32)
    for i in range(0, len(idx_all), bs):
        idx = idx_all[i:i + bs]
        x, mask, avail = to_batch(d, idx, np.tile(row, (len(idx), 1)))
        logits, gate = model(x, mask, avail, return_gate=True)
        H.append(cap["h"].double().numpy()); L.append(logits.double().numpy()); G.append(gate.double().numpy())
    hook.remove()
    return np.concatenate(H), np.concatenate(L), np.concatenate(G)

def rep_stats(Z):
    mu = Z.mean(0); C = np.cov(Z, rowvar=False); spread = float(np.sqrt(max(np.trace(C), 1e-300)))
    ev = np.clip(np.linalg.eigvalsh(C), 0, None); p = ev / max(ev.sum(), 1e-300); p = p[p > 0]
    U = Z / np.clip(np.linalg.norm(Z, axis=1, keepdims=True), 1e-300, None); n = len(Z); s = U.sum(0)
    return {"mean_norm": float(np.linalg.norm(mu)), "spread": spread, "ratio_R": float(np.linalg.norm(mu)) / spread,
            "pair_cos": float((s @ s - n) / (n * (n - 1))), "eff_rank": float(np.exp(-(p * np.log(p)).sum()))}

def readout_r(Ztr, ytr, Zva, yva):
    """Least-squares read-out with a fixed small penalty on standardised features (one linear solve). Returns Pearson r."""
    mu, sd = Ztr.mean(0), Ztr.std(0); keep = sd > 1e-9 * max(sd.max(), 1e-300)
    A, B = (Ztr[:, keep] - mu[keep]) / sd[keep], (Zva[:, keep] - mu[keep]) / sd[keep]
    w = np.linalg.solve(A.T @ A + RIDGE * np.eye(A.shape[1]), A.T @ (ytr - ytr.mean()))
    pred = B @ w + ytr.mean()
    return float(np.corrcoef(pred, yva)[0, 1]) if pred.std() > 0 else 0.0

def cos_rows(a, b):
    return float(np.mean((a * b).sum(1) / np.clip(np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-300, None)))

@torch.no_grad()
def bias_norm(model, m):
    return float(model.proj[m][1].bias.norm()) if isinstance(model, FFVar) else float(model.proj[m].bias.norm())

## 4. Run the measurements (MOSI first, then MOSEI; each result saved as it finishes)

In [5]:
REP, HROWS, CHECKS = [], [], []
for ds in ["MOSI", "MOSEI"]:
    f_rep, f_h = f"{OUT_DIR}/{ds}_representations.csv", f"{OUT_DIR}/{ds}_classifier_input.csv"
    if os.path.exists(f_rep) and os.path.exists(f_h):
        print(f"{ds}: already measured, loading saved results."); REP += pd.read_csv(f_rep).to_dict("records")
        HROWS += pd.read_csv(f_h).to_dict("records"); continue
    t0 = time.time(); lab_dir = f"{OUT_DIR}/_{ds.lower()}_labels"; os.makedirs(lab_dir, exist_ok=True)
    data = load_data(PATHS[ds], lab_dir)
    tr, va = data["train"], data["valid"]; n0 = min(256, len(va["y"]))
    va_small = {"feats": {m: va["feats"][m][:n0] for m in MODS}, "mask": va["mask"][:n0], "y": va["y"][:n0]}
    sub = np.sort(np.random.RandomState(0).permutation(len(tr["y"]))[:H_TRAIN_MAX])
    rep_rows, h_rows = [], []
    for vid in REP_MODELS:
        for s in SEEDS:
            model = load_model(ds, vid, s)
            # check 1: the loaded model reproduces the saved Notebook 8 all-inputs test predictions exactly
            saved = np.load(f"{V8}/{ds.lower()}/preds/{vid}_seed{s}.npz")["test_TpApV"]
            now = predict_cond(model, data["test"], "T+A+V")
            diff = float(np.abs(now - saved).max())
            # check 2: z_all() equals what the model passes to Eq. 15 (return_z)
            x, mask, avail = to_batch(va, np.arange(n0), np.ones((n0, 3), np.float32))
            with torch.no_grad():
                _, zr = model(x, mask, avail, return_z=True)
            za = z_all(model, va_small)
            zdiff = max(float(np.abs(zr[m].double().numpy() - za[m]).max()) for m in MODS)
            CHECKS.append({"dataset": ds, "model": vid, "seed": s, "max_diff_vs_saved_test_probs": diff, "max_diff_z": zdiff})
            assert diff < 1e-5 and zdiff < 1e-4, f"{ds} {vid} seed {s}: loaded model does not match Notebook 8 ({diff}, {zdiff})"
            Ztr, Zva = z_all(model, tr), z_all(model, va)
            for m in MODS:
                r = {"dataset": ds, "model": vid, "seed": s, "modality": m, "bias_norm": bias_norm(model, m),
                     "cos_with_text": cos_rows(Zva["text"], Zva[m]) if m != "text" else 1.0,
                     **rep_stats(Zva[m]), "readout_r": readout_r(Ztr[m], tr["score"], Zva[m], va["score"])}
                r["bias_over_spread"] = r["bias_norm"] / r["spread"]; rep_rows.append(r)
            hv_full = forward_cond(model, va, "T+A+V")[0]; mu_full, sp_full = hv_full.mean(0), rep_stats(hv_full)["spread"]
            for c in NO_TEXT + ["T+A+V"]:
                Htr, _, _ = forward_cond(model, tr, c, sub); Hva, Lva, Gva = forward_cond(model, va, c)
                st = rep_stats(Hva); pred = Lva.argmax(1)
                h_rows.append({"dataset": ds, "model": vid, "seed": s, "condition": c, "ratio_R_h": st["ratio_R"], "eff_rank_h": st["eff_rank"],
                               "readout_r_h": readout_r(Htr, tr["score"][sub], Hva, va["score"]),
                               "own_r": float(np.corrcoef(Lva[:, 2] - Lva[:, 0], va["score"])[0, 1]) if (Lva[:, 2] - Lva[:, 0]).std() > 0 else 0.0,
                               "own_rho": float(spearmanr(Lva[:, 2] - Lva[:, 0], va["score"])[0]) if (Lva[:, 2] - Lva[:, 0]).std() > 0 else 0.0,
                               "logit_std": float(Lva.std(0).mean()), "shift_from_full": float(np.linalg.norm(Hva.mean(0) - mu_full)) / sp_full,
                               "gate_text": float(Gva[:, 0].mean()), "gate_audio": float(Gva[:, 1].mean()), "gate_video": float(Gva[:, 2].mean()),
                               "valid_majority_share": float(np.bincount(pred, minlength=3).max() / len(pred))})
            print(f"  {ds} {vid} seed {s}: done (outputs match Notebook 8: max diff {diff:.1g}) | {time.time() - t0:.0f}s", flush=True)
    pd.DataFrame(rep_rows).to_csv(f_rep, index=False); pd.DataFrame(h_rows).to_csv(f_h, index=False)
    REP += rep_rows; HROWS += h_rows; del data
pd.DataFrame(CHECKS).to_csv(f"{OUT_DIR}/load_checks.csv", index=False)
REP, HROWS = pd.DataFrame(REP), pd.DataFrame(HROWS)

Loaded in 10s
Feature sizes: {'text': 768, 'audio': 5, 'vision': 20}
train:  1,284 clips | negative 552 / neutral 53 / positive 679
valid:    229 clips | negative 92 / neutral 13 / positive 124
test :    686 clips | negative 379 / neutral 30 / positive 277
  MOSI M2 seed 42: done (outputs match Notebook 8: max diff 1e-07) | 15s
  MOSI M2 seed 43: done (outputs match Notebook 8: max diff 2e-07) | 19s
  MOSI M2 seed 44: done (outputs match Notebook 8: max diff 2e-07) | 22s
  MOSI A0 seed 42: done (outputs match Notebook 8: max diff 1e-07) | 26s
  MOSI A0 seed 43: done (outputs match Notebook 8: max diff 2e-07) | 29s
  MOSI A0 seed 44: done (outputs match Notebook 8: max diff 2e-07) | 33s
  MOSI M4 seed 42: done (outputs match Notebook 8: max diff 2e-07) | 40s
  MOSI M4 seed 43: done (outputs match Notebook 8: max diff 2e-07) | 45s
  MOSI M4 seed 44: done (outputs match Notebook 8: max diff 2e-07) | 50s
  MOSI B6 seed 42: done (outputs match Notebook 8: max diff 3e-07) | 55s
  MOSI B6 see

## 5. Predicted-class shares without text (test set, from the saved Notebook 8 predictions — all 14 models)

In [6]:
PRED = []
for ds in ["MOSI", "MOSEI"]:
    y = np.load(f"{V8}/{ds.lower()}/test_labels.npy"); true = np.bincount(y, minlength=3) / len(y)
    const = {LABELS[k]: 100 * (2 * (y == k).sum() / ((y == k).sum() + len(y))) / 3 for k in range(3)}
    print(f"\n{ds} test: true shares neg/neu/pos = {true.round(3)} | macro-F1 of always predicting one class: "
          + ", ".join(f"{k} {v:.1f}" for k, v in const.items()))
    for v in ALL:
        for s in SEEDS:
            z = np.load(f"{V8}/{ds.lower()}/preds/{v}_seed{s}.npz")
            for c in NO_TEXT + ["T+A+V"]:
                sh = np.bincount(z[f"test_{c.replace('+', 'p')}"].argmax(1), minlength=3) / len(y)
                PRED.append({"dataset": ds, "model": v, "seed": s, "condition": c, "neg": sh[0], "neu": sh[1], "pos": sh[2], "majority": sh.max(),
                             "majority_class": LABELS[int(sh.argmax())]})
PRED = pd.DataFrame(PRED); PRED.to_csv(f"{OUT_DIR}/predicted_class_shares.csv", index=False)
for ds in ["MOSEI", "MOSI"]:
    t = PRED[PRED.dataset == ds].groupby(["model", "condition"], sort=False)[["neg", "neu", "pos", "majority"]].mean().unstack("condition")
    show = pd.DataFrame({c: [f"{100*t.loc[v, ('neg', c)]:.0f}/{100*t.loc[v, ('neu', c)]:.0f}/{100*t.loc[v, ('pos', c)]:.0f}" for v in ALL]
                         for c in NO_TEXT + ["T+A+V"]}, index=ALL)
    print(f"\n{ds}: predicted shares neg/neu/pos (%), test, mean of 3 seeds")
    print(show.to_string())


MOSI test: true shares neg/neu/pos = [0.552 0.044 0.404] | macro-F1 of always predicting one class: negative 23.7, neutral 2.8, positive 19.2

MOSEI test: true shares neg/neu/pos = [0.29 0.22 0.49] | macro-F1 of always predicting one class: negative 15.0, neutral 12.0, positive 21.9

MOSEI: predicted shares neg/neu/pos (%), test, mean of 3 seeds
           A         V       A+V     T+A+V
M2  20/53/27  24/47/29  24/45/31  32/25/44
M3  29/28/43  31/37/33  31/32/37  32/21/47
M4  29/27/44  28/31/40  27/26/47  29/25/46
A0    1/98/1    6/89/5    7/86/8  33/24/43
A1  25/43/32  27/42/30  24/44/32  32/25/44
A2  30/26/44  36/24/40  34/25/41  32/25/43
A3  28/27/45  25/41/34  26/33/42  32/23/45
A4  16/57/27  16/58/26  19/52/29  30/26/44
B1  29/24/47  34/32/34  35/27/38  32/28/40
B2  29/29/42  33/28/40  26/28/46  32/26/42
B3  28/27/44  37/27/36  33/22/45  31/25/44
B4  28/30/43  24/38/38  25/33/41  30/23/46
B5  33/24/42  30/30/40  30/24/46  30/23/47
B6  29/21/50  41/21/38  35/20/45  34/22/44

MOSI:

## 6. Results: representations (validation set, mean of 3 seeds) and the classifier input

In [7]:
pd.set_option("display.width", 250)
cols = ["bias_norm", "spread", "bias_over_spread", "ratio_R", "pair_cos", "eff_rank", "cos_with_text", "readout_r"]
for ds in ["MOSEI", "MOSI"]:
    t = REP[(REP.dataset == ds) & (REP.modality != "text")].groupby(["modality", "model"], sort=False)[cols].mean()
    print(f"\n{ds}: audio/video representations z_m (mean of 3 seeds)"); print(t.round(3).to_string())
    tt = REP[(REP.dataset == ds) & (REP.modality == "text")].groupby("model", sort=False)[["bias_norm", "spread", "ratio_R", "eff_rank", "readout_r"]].mean()
    print(f"{ds}: text representation z_t (reference)"); print(tt.round(3).to_string())
    h = HROWS[HROWS.dataset == ds].groupby(["condition", "model"], sort=False)[
        ["gate_text", "gate_audio", "gate_video", "shift_from_full", "ratio_R_h", "eff_rank_h", "logit_std", "readout_r_h", "own_r", "own_rho", "valid_majority_share"]].mean()
    print(f"\n{ds}: classifier input h by input combination (validation, mean of 3 seeds)"); print(h.round(3).to_string())


MOSEI: audio/video representations z_m (mean of 3 seeds)
                bias_norm  spread  bias_over_spread  ratio_R  pair_cos  eff_rank  cos_with_text  readout_r
modality model                                                                                            
audio    M2         0.842   3.395             0.248    0.267     0.078    26.376         -0.010      0.169
vision   M2         1.179   4.758             0.248    0.266     0.084    13.512         -0.015      0.203
audio    A0         2.530   0.991             2.562    2.760     0.956     6.002          0.959      0.169
vision   A0         3.350   0.988             3.421    3.375     0.932    16.281          0.945      0.203
audio    M4         0.269   6.017             0.045    0.287     0.083    26.543          0.011      0.163
vision   M4         0.295   8.456             0.035    0.324     0.093    14.208          0.006      0.206
audio    B6         2.439   3.548             0.689    3.197     0.909    27.955      

## 7. Pre-defined criteria K1–K5 and the reading

In [8]:
def criteria(ds, a, b):
    R = REP[REP.dataset == ds].set_index(["model", "seed", "modality"]); out = {}
    per = lambda f: all(f(s, m) for s in SEEDS for m in ["audio", "vision"])
    out["K1 strong alignment"] = per(lambda s, m: R.loc[(a, s, m), "cos_with_text"] >= 0.90)
    out["K2 directional collapse"] = per(lambda s, m: R.loc[(a, s, m), "ratio_R"] >= 2 * R.loc[(b, s, m), "ratio_R"])
    out["K3 dimensional collapse"] = per(lambda s, m: R.loc[(a, s, m), "eff_rank"] <= 0.5 * R.loc[(b, s, m), "eff_rank"])
    ev = [(s, m) for s in SEEDS for m in ["audio", "vision"] if R.loc[(b, s, m), "readout_r"] >= 0.10]
    out["K4 information loss"] = (all(R.loc[(a, s, m), "readout_r"] <= 0.8 * R.loc[(b, s, m), "readout_r"] for s, m in ev)
                                  if ev else "not evaluable (reference r < 0.10)")
    P = PRED[(PRED.dataset == ds) & PRED.condition.isin(NO_TEXT)].groupby(["model", "condition"])["majority"].mean()
    out["K5 prediction collapse"] = all(P.loc[(a, c)] >= 0.90 for c in NO_TEXT) and all(P.loc[(b, c)] < 0.90 for c in NO_TEXT)
    HH = HROWS[(HROWS.dataset == ds) & HROWS.condition.isin(NO_TEXT)].groupby(["model", "condition"])[["own_r", "own_rho"]].mean()
    H = HH["own_r"]
    ev6 = [c for c in NO_TEXT if H.loc[(b, c)] >= 0.10]
    out["K6 ranking kept"] = all(H.loc[(a, c)] >= 0.8 * H.loc[(b, c)] for c in ev6) if ev6 else "not evaluable (reference r < 0.10)"
    rank_txt = "; ".join(f"{c}: Pearson {a} {HH.loc[(a, c), 'own_r']:.3f} vs {b} {HH.loc[(b, c), 'own_r']:.3f}, "
                         f"Spearman {a} {HH.loc[(a, c), 'own_rho']:.3f} vs {b} {HH.loc[(b, c), 'own_rho']:.3f}" for c in NO_TEXT)
    out["K6 values (validation)"] = rank_txt
    k = {kk.split()[0]: v for kk, v in out.items() if not kk.startswith("K6 values")}
    if not k["K1"]:
        reading = "Alignment is not strong (K1 fails): the collapse hypothesis does not apply."
    elif (k["K2"] or k["K3"]) and k["K4"] is True and k["K5"]:
        reading = "Consistent with a representation collapse that removes audio/video sentiment information (association, not proven cause)."
    elif (k["K2"] or k["K3"]) and k["K4"] is False and k["K5"]:
        reading = ("Representations collapse in direction/dimension and predictions without text become near-constant. Substantial loss of "
                   "linearly decodable sentiment information in z was NOT established (K4 fails). This does not show that the information "
                   "is preserved; for M2/A0 this test can only detect a near-singular projection (association only).")
    elif (k["K2"] or k["K3"]) and isinstance(k["K4"], str) and k["K5"]:
        reading = ("Representations collapse in direction/dimension and predictions without text become near-constant. Information loss "
                   "could not be assessed: K4 is not evaluable because the reference read-out r is below 0.10 (association only).")
    elif not k["K2"] and not k["K3"]:
        reading = "No directional or dimensional collapse found: the collapse hypothesis is not supported."
    else:
        reading = "Mixed pattern: see the tables; no single reading applies."
    if k["K1"] and k["K5"] and k["K6"] is True:
        reading += (" Added clause (K5 and K6, Pearson criterion met; Spearman reported alongside): the model's own outputs still track "
                    "sentiment without text, so the near-constant predictions are consistent with an output shift towards one class rather "
                    "than with lost ranking information. Ranking values: " + rank_txt + ".")
    elif k["K1"] and k["K5"] and isinstance(k["K6"], str):
        reading += " K6 is not evaluable (reference r < 0.10), so whether ranking is kept is undetermined."
    return out, reading

VERDICT = {}
for ds in ["MOSEI", "MOSI"]:
    for a, b in PAIRS:
        out, reading = criteria(ds, a, b)
        tag = "PRIMARY" if (ds, a) == ("MOSEI", "A0") else "secondary"
        print(f"\n[{tag}] {ds}: {a} vs {b}")
        for kk, v in out.items():
            print(f"  {kk:26s}: {v}")
        print("  Reading:", reading)
        VERDICT[f"{ds}_{a}_vs_{b}"] = {"criteria": {kk: (v if isinstance(v, str) else bool(v)) for kk, v in out.items()}, "reading": reading}
json.dump({"verdicts": VERDICT, "ridge": RIDGE, "h_train_max": H_TRAIN_MAX, "seeds": SEEDS,
           "note": "Associations only; no causal test (would need retraining with different lambda1)."},
          open(f"{OUT_DIR}/EQ15_CHECK.json", "w"), indent=2)
print(f"\nSaved to {OUT_DIR}: MOSEI/MOSI_representations.csv, *_classifier_input.csv, predicted_class_shares.csv, load_checks.csv, EQ15_CHECK.json")


[PRIMARY] MOSEI: A0 vs M2
  K1 strong alignment       : True
  K2 directional collapse   : True
  K3 dimensional collapse   : False
  K4 information loss       : False
  K5 prediction collapse    : False
  K6 ranking kept           : True
  K6 values (validation)    : A: Pearson A0 0.132 vs M2 0.093, Spearman A0 0.130 vs M2 0.093; V: Pearson A0 0.173 vs M2 0.188, Spearman A0 0.183 vs M2 0.200; A+V: Pearson A0 0.205 vs M2 0.193, Spearman A0 0.218 vs M2 0.193
  Reading: Mixed pattern: see the tables; no single reading applies.

[secondary] MOSEI: B6 vs M4
  K1 strong alignment       : True
  K2 directional collapse   : True
  K3 dimensional collapse   : False
  K4 information loss       : False
  K5 prediction collapse    : False
  K6 ranking kept           : True
  K6 values (validation)    : A: Pearson B6 0.150 vs M4 0.180, Spearman B6 0.127 vs M4 0.173; V: Pearson B6 0.223 vs M4 0.218, Spearman B6 0.209 vs M4 0.214; A+V: Pearson B6 0.239 vs M4 0.239, Spearman B6 0.229 vs M4 0.227
  R